In [1]:
from google.colab import userdata
from huggingface_hub import login
from transformers import pipeline
from PIL import Image
import requests

# Retrieve the token from Colab Secrets
try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print("Successfully logged into Hugging Face Hub!")
except userdata.SecretNotFoundError:
    print("Error: HF_TOKEN not found in Secrets. Please add it via the key icon on the left.")

Successfully logged into Hugging Face Hub!


In [ ]:
pip install -U transformers

In [14]:
# ─────────────────────────────────────────────
# 1. SENTIMENT ANALYSIS
# ─────────────────────────────────────────────
print("\n" + "="*60)
print("1. SENTIMENT ANALYSIS")
print("="*60)

sentiment = pipeline("sentiment-analysis", model="distilbert/distilbert-base-uncased-finetuned-sst-2-english")
sentences = [
    "I've been not waiting for a EE471 course my whole life.",
    "I hate EE471 course"
]
results = sentiment(sentences)
for s, r in zip(sentences, results):
    print(f"  Text   : {s}")
    print(f"  Result : {r['label']} (score: {r['score']:.4f})\n")



1. SENTIMENT ANALYSIS


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

  Text   : I've been not waiting for a EE471 course my whole life.
  Result : POSITIVE (score: 0.6272)

  Text   : I hate EE471 course
  Result : NEGATIVE (score: 0.9993)



In [3]:
# ─────────────────────────────────────────────
# 2. ZERO-SHOT CLASSIFICATION
# ─────────────────────────────────────────────
print("="*60)
print("2. ZERO-SHOT CLASSIFICATION")
print("="*60)

zero_shot = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
text = "Berkshire keeps their cash reserves at an extremely high level."
candidate_labels = ["finance", "politics", "technology", "sports", "economics"]
result = zero_shot(text, candidate_labels=candidate_labels)
print(f"  Text   : {text}")
for label, score in zip(result["labels"], result["scores"]):
    print(f"  {label:<15}: {score:.4f}")

2. ZERO-SHOT CLASSIFICATION


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

  Text   : Berkshire keeps their cash reserves at an extremely high level.
  finance        : 0.8234
  economics      : 0.1055
  technology     : 0.0419
  sports         : 0.0160
  politics       : 0.0132


In [4]:
# ─────────────────────────────────────────────
# 3. TEXT GENERATION (from incomplete sentence)
# ─────────────────────────────────────────────
print("\n" + "="*60)
print("3. TEXT GENERATION")
print("="*60)

generator = pipeline("text-generation", model="gpt2")
prompt = "If I continue to successfully complete all in-class exercises in EE471 course,"

# Adding sampling and repetition penalty to prevent the looping issue you saw
results = generator(
    prompt,
    max_length=100,
    num_return_sequences=2,
    truncation=True,
    do_sample=True,           # Enable sampling instead of greedy search
    top_p=0.95,               # Use nucleus sampling for better diversity
    temperature=0.7,          # Adjust randomness (0.7 is a good balance)
    no_repeat_ngram_size=3    # Prevents the model from repeating 3-word phrases
)

print(f"  Prompt : {prompt}")
for i, r in enumerate(results, 1):
    print(f"  Option {i}: {r['generated_text']}\n")


3. TEXT GENERATION


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'temperature', 'num_return_sequences', 'max_length', 'top_p', 'do_sample', 'no_repeat_ngram_size'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Prompt : If I continue to successfully complete all in-class exercises in EE471 course,
  Option 1: If I continue to successfully complete all in-class exercises in EE471 course, then I will be able to continue on to my next course, which I am currently completing in the second year of my EE program."

According to the report, the students have received "no complaints about any of the exercises being performed in the first year of the program".

The students have already completed the EE471 courses in all three EE classes, and they were able to complete their second year at the same time.

However, the report said that the students are now unable to complete the courses in the same year.
.

  Option 2: If I continue to successfully complete all in-class exercises in EE471 course, I will then have the opportunity to complete my entire course on an integrated basis, which will allow me to focus my energy on the next level of EE471 training and practice."

"The success of the course wil

In [5]:
# ─────────────────────────────────────────────
# 4. MASK FILLING
# ─────────────────────────────────────────────
print("\n" + "="*60)
print("4. MASK FILLING")
print("="*60)

# BERT uses [MASK], RoBERTa uses <mask>
unmasker = pipeline("fill-mask", model="bert-base-uncased")
masked_sentence = "To understand generative AI, one must study [MASK] well."
results = unmasker(masked_sentence)
print(f"  Masked : {masked_sentence}")
print("  Top predictions:")
for r in results[:3]:
    print(f"    → '{r['token_str']}' (score: {r['score']:.4f})  →  {r['sequence']}")



4. MASK FILLING


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

  Masked : To understand generative AI, one must study [MASK] well.
  Top predictions:
    → 'it' (score: 0.3629)  →  to understand generative ai, one must study it well.
    → 'them' (score: 0.3304)  →  to understand generative ai, one must study them well.
    → 'as' (score: 0.0313)  →  to understand generative ai, one must study as well.


In [6]:
import requests
from PIL import Image
from transformers import pipeline

# ─────────────────────────────────────────────
# 5. NAMED ENTITY RECOGNITION (NER)
# ─────────────────────────────────────────────
print("\n" + "="*60)
print("5. NAMED ENTITY RECOGNITION")
print("="*60)

ner = pipeline("ner", model="dslim/bert-base-NER", aggregation_strategy="simple")
ner_sentence = (
    "I am Nate, a research assistant in Izmir Institute of Technology, "
    "and currently living and working in beautiful city İzmir in Türkiye."
)
entities = ner(ner_sentence)

# Post-processing to merge contiguous sub-word tokens of the same entity group
processed_entities = []
if entities:
    current_entity = entities[0]
    for i in range(1, len(entities)):
        next_entity = entities[i]
        # Check if the next entity immediately follows the current one
        # and if they belong to the same entity group.
        # This handles cases like 'T' and '##ürkiye' where the 'simple' aggregation
        # might not combine them for some reason.
        if (next_entity['start'] == current_entity['end'] and
            next_entity['entity_group'] == current_entity['entity_group']):
            # Merge them
            # Remove '##' from the beginning of the next word part before concatenating
            current_entity['word'] += next_entity['word'].replace('##', '')
            current_entity['end'] = next_entity['end']
            # Average score or take max/min; averaging is a common heuristic
            current_entity['score'] = (current_entity['score'] + next_entity['score']) / 2
        else:
            processed_entities.append(current_entity)
            current_entity = next_entity
    processed_entities.append(current_entity) # Add the last processed entity
else:
    processed_entities = entities # If no entities, keep it empty

print(f"  Text: {ner_sentence}\n")
print("  Extracted entities (post-processed for contiguous sub-words):")
for e in processed_entities:
    # Ensure the word doesn't start with '##' after merging, if it was the first part of a merge
    display_word = e['word'].lstrip('##')
    print(f"    [{e['entity_group']}] {display_word}  (score: {e['score']:.4f})")


5. NAMED ENTITY RECOGNITION


config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

  Text: I am Nate, a research assistant in Izmir Institute of Technology, and currently living and working in beautiful city İzmir in Türkiye.

  Extracted entities (post-processed for contiguous sub-words):
    [PER] Nate  (score: 0.9988)
    [ORG] Izmir Institute of Technology  (score: 0.9955)
    [LOC] İzmir  (score: 0.9886)
    [LOC] Türkiye  (score: 0.9173)


In [7]:
# ─────────────────────────────────────────────
# 6. QUESTION ANSWERING (validate NER results)
# ─────────────────────────────────────────────
print("\n" + "="*60)
print("6. QUESTION ANSWERING (validating NER)")
print("="*60)

qa_pipeline = pipeline("question-answering", model="deepset/roberta-base-squad2")

context = ner_sentence
questions = [
    "What is the name of the person?",
    "Which organization does the person work for?",
    "Which city does the person live in?"
]

for q in questions:
    result = qa_pipeline(question=q, context=context)
    print(f"  Q: {q}")
    print(f"  A: {result['answer']}  (score: {result['score']:.4f})\n")



6. QUESTION ANSWERING (validating NER)


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/496M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaForQuestionAnswering LOAD REPORT from: deepset/roberta-base-squad2
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/79.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

  Q: What is the name of the person?
  A: Nate  (score: 0.8180)

  Q: Which organization does the person work for?
  A: Izmir Institute of Technology  (score: 0.9611)

  Q: Which city does the person live in?
  A: İzmir in Türkiye  (score: 0.2922)



In [8]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ─────────────────────────────────────────────
# 7. TEXT SUMMARIZATION
# ─────────────────────────────────────────────
print("="*60)
print("7. TEXT SUMMARIZATION")
print("="*60)

long_text = (
    "The 2008 Global Financial Crisis stands as the most severe economic collapse of the 21st century, "
    "often compared to the Great Depression of the 1930s. Triggered by the bursting of the United States "
    "housing bubble, its effects rippled across the globe, leading to the collapse of major financial "
    "institutions and a deep international recession. The crisis began with the subprime mortgage market. "
    "In the early 2000s, low interest rates and a push for homeownership led banks to issue high-risk loans "
    "to borrowers with poor credit."
)

model_name = "facebook/bart-large-cnn"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Tokenize input text
inputs = tokenizer([long_text], max_length=1024, return_tensors='pt', truncation=True)

# Generate summary
summary_ids = model.generate(
    inputs['input_ids'],
    num_beams=4,
    max_length=80,
    min_length=30,
    early_stopping=True
)

summary_text = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print(f"  Original ({len(long_text)} chars)")
print(f"  Summary : {summary_text}")

7. TEXT SUMMARIZATION


config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

  Original (529 chars)
  Summary : The 2008 Global Financial Crisis stands as the most severe economic collapse of the 21st century. Triggered by the bursting of the U.S. housing bubble, its effects rippled across the globe.


In [9]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ─────────────────────────────────────────────
# 8. TRANSLATION (English → Turkish)
# ─────────────────────────────────────────────
print("\n" + "="*60)
print("8. TRANSLATION (English → Turkish)")
print("="*60)

# Using a different reliable English-Turkish model repository
model_name = "Helsinki-NLP/opus-mt-tc-big-en-tr"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

text_to_translate = (
    "The 2008 Global Financial Crisis stands as the most severe economic collapse "
    "of the 21st century, often compared to the Great Depression."
)

inputs = tokenizer(text_to_translate, return_tensors="pt")
translated_tokens = model.generate(**inputs)
translated_text = tokenizer.decode(translated_tokens[0], skip_special_tokens=True)

print(f"  EN: {text_to_translate}")
print(f"  TR: {translated_text}")


8. TRANSLATION (English → Turkish)


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/337 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/797k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/833k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/470M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

  EN: The 2008 Global Financial Crisis stands as the most severe economic collapse of the 21st century, often compared to the Great Depression.
  TR: 2008 Küresel Finansal Krizi, 21. yüzyılın en şiddetli ekonomik çöküşü olarak, genellikle Büyük Buhran ile karşılaştırıldığında durmaktadır.


In [10]:
# ─────────────────────────────────────────────
# 9. IMAGE CLASSIFICATION (Google ViT)
# ─────────────────────────────────────────────
print("\n" + "="*60)
print("9. IMAGE CLASSIFICATION (google/vit-base-patch16-224)")
print("="*60)

# Note: use_fast=True is set, though transformers may still log a warning depending on the local env
image_classifier = pipeline("image-classification", model="google/vit-base-patch16-224", use_fast=True)

# Using an alternative URL (Hugging Face's own sample cat image) to bypass connection issues
img_url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/cats.png"
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

try:
    response = requests.get(img_url, stream=True, headers=headers, timeout=10)
    response.raise_for_status()
    image = Image.open(response.raw).convert("RGB")

    predictions = image_classifier(image)
    print(f"  Image URL: {img_url}")
    print("  Top predictions:")
    for p in predictions[:3]:
        print(f"    {p['label']:<30} score: {p['score']:.4f}")
except Exception as e:
    print(f"  Error loading or processing image: {e}")


9. IMAGE CLASSIFICATION (google/vit-base-patch16-224)


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


  Image URL: https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/cats.png
  Top predictions:
    tabby, tabby cat               score: 0.2769
    tiger cat                      score: 0.2764
    Egyptian cat                   score: 0.1403


In [11]:
import torch
import gc

# Helper to clear out GPU memory from previous tasks
def clear_gpu():
    # Delete large model variables if they exist
    global generator, unmasker, ner, qa_pipeline, model, image_classifier, asr

    # List of models to clear
    vars_to_clear = ['generator', 'unmasker', 'ner', 'qa_pipeline', 'model', 'image_classifier', 'asr']
    for var in vars_to_clear:
        if var in globals():
            del globals()[var]

    gc.collect()
    torch.cuda.empty_cache()
    print("GPU memory cleared.")

clear_gpu()

GPU memory cleared.


In [13]:
# 10. AUTOMATIC SPEECH RECOGNITION (Switching to whisper-medium to save memory)
print("\n" + "="*60)
print("10. AUTOMATIC SPEECH RECOGNITION (openai/whisper-medium)")
print("="*60)

# Using whisper-medium instead of large-v3 to avoid OOM
asr = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-medium",
    chunk_length_s=30,
    device=0
)

audio_url = "https://huggingface.co/datasets/Narsil/asr_dummy/resolve/main/1.flac"

try:
    response = requests.get(audio_url, timeout=10)
    response.raise_for_status()
    result = asr(response.content)
    print(f"  Audio Source: {audio_url}")
    print(f"  Transcription: {result['text']}")
except Exception as e:
    print(f"  Error processing audio: {e}")


10. AUTOMATIC SPEECH RECOGNITION (openai/whisper-medium)


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.06G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/947 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

...

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

preprocessor_config.json: 0.00B [00:00, ?B/s]

Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <c

  Audio Source: https://huggingface.co/datasets/Narsil/asr_dummy/resolve/main/1.flac
  Transcription:  He hoped there would be stew for dinner, turnips and carrots and bruised potatoes and fat mutton-pieces to be ladled out in thick, peppered, flour-fattened sauce.
